# Assignment 2A — Prompting as System Design

**Group 172**

| Member | BITS ID | Contribution |
|---|---|---|
| Yeldona Desai V Vittal | 2025AE05316 | 100% |
| Naveen K. B | — | 100% |
| Naman Aryan | — | 100% |
| Patil Manaliben Dipakbhai | — | 100% |

**Reused from Assignment 1A:** domain corpus (13 cleaned source papers, Variant 1 — scientific research papers), instruction dataset, and the base model `HuggingFaceTB/SmolLM2-360M` with a continual-pre-training (CPT) + QLoRA fine-tuning pipeline.

**Generator for 2A:** the 1A pipeline (CPT on the domain corpus, then QLoRA instruction fine-tuning), greedy decoding, `max_new_tokens=200`. The original 1A adapter weights were lost (ephemeral Colab storage) and have been retrained here following 1A's own code faithfully, with one deviation noted and justified in Step 1 below.

**Judge for Part A:** `Qwen2.5-7B-Instruct`, deterministic decoding.

**Runtime:** this notebook was developed and validated on a local CPU machine at reduced (dev) scale; the full-fidelity retrain (10 CPT epochs, 8 SFT epochs, matching 1A's hyperparameters exactly) is run separately on the BITS Prayogshala lab machine, and the resulting adapter is substituted in before final submission.


## Setup — Colab bootstrap

Running this notebook via `colab.research.google.com/github/.../Assignment2A_Group172.ipynb` only loads this file — not the rest of the repo (`instruction_dataset.jsonl`, `domain_corpus/`, `benchmark_10.json`, `train_adapter.py`) that the later cells need on disk next to it. This cell detects Colab, clones the repo, and installs the GPU dependency set. On a local machine or a labs box where the repo is already cloned and the environment already set up (per `RUNBOOK.md`), it does nothing.


In [ ]:
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/VijayYD/2025AE05316-LLM-2A.git"
    REPO_DIR = "2025AE05316-LLM-2A"
    if not os.path.isdir(REPO_DIR):
        print(f"Cloning {REPO_URL} ...")
        get_ipython().system(f"git clone {REPO_URL}")
    os.chdir(REPO_DIR)
    print("Installing dependencies from requirements-gpu.txt ...")
    get_ipython().system("pip install -q -r requirements-gpu.txt")
    print("Colab bootstrap done. Working directory:", os.getcwd())
else:
    print("Not running in Colab -- skipping clone/install (assumes this notebook's own directory already has the repo files).")


## Setup — Retrain the adapter for this run

The dev-scale placeholder weights are intentionally **not** committed to the repo (they are regenerated fresh, and the full-fidelity CPT weights alone are ~1.3GB, over GitHub's per-file limit). This cell checks for a CUDA GPU:

- **GPU present** (labs, Colab T4+): runs `train_adapter.py --mode full`, reproducing Assignment 1A's exact two-stage pipeline (10 CPT epochs, 8 SFT epochs on the full dataset) — this is the run that should back the real submission numbers.
- **No GPU** (this local Mac): runs `train_adapter.py --mode dev`, a fast truncated run used only to validate that the rest of this notebook's pipeline runs end to end. Any generations produced under the dev-scale adapter are expected to be degenerate/undertrained and are documented as such wherever they appear below.


In [ ]:
import torch

if torch.cuda.is_available():
    print(f"CUDA GPU detected ({torch.cuda.get_device_name(0)}) -- running the full-fidelity retrain.")
    print("python train_adapter.py --mode full")
    get_ipython().system("python train_adapter.py --mode full")
    RETRAIN_MODE = "full"
else:
    print("No CUDA GPU detected -- running the fast dev-scale retrain for local pipeline validation only.")
    print("python train_adapter.py --mode dev")
    get_ipython().system("python train_adapter.py --mode dev")
    RETRAIN_MODE = "dev"

print("\nRETRAIN_MODE =", RETRAIN_MODE)


## Step 0 — Environment check

In [1]:
import sys, platform
import torch
import transformers
import peft

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Selected device:", DEVICE)


Python: 3.11.10
Platform: macOS-26.6.2-x86_64-i386-64bit
torch: 2.2.2
transformers: 4.46.3
peft: 0.13.2
CUDA available: False
MPS available: True
Selected device: mps


## Step 1 — Dataset

`instruction_dataset.jsonl` (58 pairs): the 50 original Assignment 1A pairs, kept unchanged so the retrained adapter stays faithful to 1A's training data, plus 8 new pairs added afterward (approved) to broaden coverage of under-represented source papers. Audited below against the quality rules: minimum pair count, a response-length filter (>= 30 words), question-type and per-paper coverage, template-repetition flagging (> 20% of pairs), and a verbatim-copy check against the source papers.


In [ ]:
import json
import re
from collections import Counter

with open("instruction_dataset.jsonl") as f:
    dataset_pairs = [json.loads(l) for l in f]

print(f"Total pairs: {len(dataset_pairs)} (minimum required: 40-50)")
print(f"Field names: {sorted(dataset_pairs[0].keys())}")


### Response-length filter (>= 30 words)


In [ ]:
def word_count(s):
    return len(s.split())

lengths = [word_count(p["response"]) for p in dataset_pairs]
short_pairs = [(i, n) for i, n in enumerate(lengths) if n < 30]

print(f"Pairs failing the >= 30-word filter: {len(short_pairs)}/{len(dataset_pairs)}")
print(f"Response word count: min={min(lengths)} max={max(lengths)} mean={sum(lengths)/len(lengths):.1f}")
print("\nNote: most failures are among the original 50 (1A) pairs, kept unchanged for retraining "
      "fidelity per the earlier approved decision; the 8 supplementary pairs were written to pass "
      "this filter.")


### Question-type and per-paper coverage


In [ ]:
def classify_type(instruction):
    t = instruction.lower()
    if any(k in t for k in ["compare", " vs ", "versus", "difference between", "differ from", "compared to", "contrast"]):
        return "comparative"
    if any(k in t for k in ["how do", "how does one", "how to", "steps", "procedure", "process of", "walk through", "implement"]):
        return "procedural"
    return "factual"

types = [classify_type(p["instruction"]) for p in dataset_pairs]
print("Question-type distribution:", Counter(types))

paper_keywords = {
    "bert": ["bert", "masked language model", "mlm", "next sentence prediction", "roberta"],
    "gpt3": ["gpt-3", "gpt3", "few-shot learning", "in-context learning"],
    "llama": ["llama", "rmsnorm", "swiglu", "rotary", "rope"],
    "chinchilla": ["chinchilla", "compute-optimal", "scaling law"],
    "transformer_xl": ["transformer-xl", "transformer xl", "segment-level recurrence", "relative positional"],
    "qlora": ["qlora", "nf4", "normalfloat", "4-bit", "double quantization", "paged optimizer"],
    "prefix_tuning": ["prefix-tuning", "prefix tuning", "virtual tokens"],
    "prompt_tuning": ["prompt tuning", "soft prompt"],
    "adapters": ["adapter module", "adapter layer", "bottleneck adapter"],
    "flash_attention": ["flashattention", "flash attention", "tiling", "io-aware"],
    "dpo": ["dpo", "direct preference optimization", "bradley-terry"],
    "instructgpt": ["instructgpt", "rlhf", "reward model", "human feedback"],
    "rag": ["retrieval-augmented", "rag", "retriever", "dense passage"],
    "lora_generic": ["lora", "low-rank adaptation", "low rank adaptation"],
}

def attribute_paper(instruction, response):
    text = (instruction + " " + response).lower()
    return [paper for paper, kws in paper_keywords.items() if any(kw in text for kw in kws)]

attributions = [attribute_paper(p["instruction"], p["response"]) for p in dataset_pairs]
paper_counts = Counter(paper for a in attributions for paper in a)
unattributed = [i for i, a in enumerate(attributions) if not a]

print("\nPer-paper keyword coverage:", paper_counts)
print(f"Pairs with no keyword match to a known paper/topic: {len(unattributed)}")


### Template-repetition flag (> 20% of pairs)


In [ ]:
def leading_template(instruction):
    return " ".join(instruction.strip().split()[:3]).lower().rstrip("?,.")

templates = [leading_template(p["instruction"]) for p in dataset_pairs]
template_counts = Counter(templates)
top_template, top_count = template_counts.most_common(1)[0]
top_pct = 100 * top_count / len(dataset_pairs)

print(f"Most frequent leading phrase: {top_template!r} -- {top_count}/{len(dataset_pairs)} pairs ({top_pct:.1f}%)")
print("Flag (> 20%):", "YES" if top_pct > 20 else "no")
print("\nTop 5 leading phrases:")
for tmpl, cnt in template_counts.most_common(5):
    print(f"  {cnt:3d} ({100*cnt/len(dataset_pairs):4.1f}%)  {tmpl!r}")


### Verbatim-copy check (>= 12 consecutive words shared with a source paper)


In [ ]:
from pathlib import Path

corpus_dir = Path("domain_corpus")
corpus_texts = {f.stem: f.read_text(encoding="utf-8", errors="ignore") for f in corpus_dir.glob("*.txt")}

def normalize(text):
    return re.sub(r"\s+", " ", text.lower()).strip()

def ngrams(words, n=12):
    return set(tuple(words[i:i + n]) for i in range(len(words) - n + 1))

corpus_ngrams = {name: ngrams(normalize(text).split()) for name, text in corpus_texts.items()}

verbatim_hits = []
for i, p in enumerate(dataset_pairs):
    resp_words = normalize(p["response"]).split()
    if len(resp_words) < 12:
        continue
    resp_ngrams = ngrams(resp_words)
    for name, cng in corpus_ngrams.items():
        shared = resp_ngrams & cng
        if shared:
            verbatim_hits.append((i, name, len(shared)))

print(f"Responses with a >= 12-word verbatim span shared with a source paper: {len(verbatim_hits)}/{len(dataset_pairs)}")
for i, name, n in verbatim_hits[:10]:
    print(f"  idx {i} vs {name}: {n} shared 12-grams")


### Spot-check sample (15 pairs, fixed seed)


In [ ]:
import random

random.seed(42)
sample_idx = random.sample(range(len(dataset_pairs)), 15)

for i in sample_idx:
    print(f"\n[{i}] Q: {dataset_pairs[i]['instruction']}")
    print(f"     A: {dataset_pairs[i]['response']}")


TODO (Vijay): spot-check the 15 sampled pairs above -- confirm each response is a concise, accurate answer in the dataset's own words (not text copied verbatim from a paper), and record your verdict here.


## Step 1 — Load the fine-tuned generator (1A pipeline reproduction)

The generator is `HuggingFaceTB/SmolLM2-360M` after Assignment 1A's two-stage pipeline: continual pre-training (CPT) on the domain corpus, then QLoRA instruction fine-tuning on `instruction_dataset.jsonl`.

**Deviation from 1A's literal code, and why:** 1A's `tokenize_sft` function set `labels = input_ids.copy()` without masking padding positions. Since prompts are padded to `max_length=512` while real content is far shorter, this computes cross-entropy loss mostly over padding tokens, pushing the loss toward `ln(vocab_size) ≈ 10.8` regardless of real learning and destabilizing training (verified: reproducing 1A's literal code on its own first training batch gives loss ≈10.8–11, not the ≈7.5 shown in 1A's stored notebook output — the two are inconsistent). The fix (masking padding positions with `-100` in the labels) was verified to restore a stable, sane loss trajectory and is applied in `train_adapter.py`. No other hyperparameters were changed.

**Scale note:** the adapter loaded below is from a *dev-scale* run (40 packed CPT sequences, 15 max steps per stage) used only to validate this pipeline end to end on a CPU machine. It is a placeholder — the full-fidelity adapter (1A's exact 10 CPT / 8 SFT epochs on the complete corpus and dataset) is trained separately on Prayogshala and substituted here before final submission, by pointing `CPT_MODEL_PATH` / `ADAPTER_PATH` at the new weights.


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL_ID = "HuggingFaceTB/SmolLM2-360M"
CPT_MODEL_PATH = "./cpt_final_model"
ADAPTER_PATH = "./final_qlora_adapter"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(CPT_MODEL_PATH, torch_dtype=torch.float32)
generator_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
generator_model.to(DEVICE)
generator_model.eval()

n_trainable = sum(p.numel() for p in generator_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in generator_model.parameters())
print(f"Loaded generator: {BASE_MODEL_ID} + CPT + QLoRA adapter")
print(f"Trainable params (should be 0 in eval/inference): {n_trainable} / {n_total}")


Loaded generator: HuggingFaceTB/SmolLM2-360M + CPT + QLoRA adapter
Trainable params (should be 0 in eval/inference): 0 / 363459520


## Step 2 — Generator wrapper, per-call helper, and the pattern-function contract

- `generate(prompt, **overrides)` — raw wrapper around the model: builds the 1A-style instruction prompt, runs greedy decoding with `max_new_tokens=200` by default, returns the decoded continuation text only.
- `call_generator(prompt, **overrides)` — per-call helper used by every prompt pattern: wraps `generate`, measuring prompt tokens, generated tokens, and wall-clock latency for that one call.
- **Pattern function contract:** every prompt pattern implemented in Parts A/B/C is a plain Python function `pattern(query) -> (answer, tokens_used, latency_sec)`, built on top of one or more `call_generator` calls. `tokens_used` is the sum of prompt + generated tokens across all model calls the pattern makes (multi-call patterns such as self-critique or ReAct make several), and `latency_sec` is the pattern's total wall-clock time.


In [3]:
import time

GEN_DEFAULTS = dict(max_new_tokens=200, do_sample=False)

PROMPT_TEMPLATE = "### Instruction:\n{instruction}\n\n### Response:\n"

def generate(prompt: str, **overrides) -> str:
    """Raw generation call: formats `prompt` as a 1A-style instruction, greedily decodes, returns the continuation text (prompt stripped, stops at the first new '### Instruction:' if the model keeps going)."""
    settings = {**GEN_DEFAULTS, **overrides}
    full_prompt = PROMPT_TEMPLATE.format(instruction=prompt)
    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = generator_model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            **settings,
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    text = text.split("### Instruction:")[0].strip()
    return text

def call_generator(prompt: str, **overrides):
    """Per-call helper: wraps generate(), returns (text, prompt_tokens, generated_tokens, latency_sec)."""
    prompt_tokens = len(tokenizer(PROMPT_TEMPLATE.format(instruction=prompt), add_special_tokens=False)["input_ids"])
    t0 = time.perf_counter()
    text = generate(prompt, **overrides)
    latency_sec = time.perf_counter() - t0
    generated_tokens = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    return text, prompt_tokens, generated_tokens, latency_sec

print("Generation defaults:", GEN_DEFAULTS)
print("Pattern contract: pattern(query) -> (answer, tokens_used, latency_sec)")


Generation defaults: {'max_new_tokens': 200, 'do_sample': False}
Pattern contract: pattern(query) -> (answer, tokens_used, latency_sec)


## Step 2 — Fixed benchmark set

Loaded from `benchmark_10.json`: 10 queries drawn from the instruction dataset via stratified random sampling (seed=42, 5 factual / 3 procedural / 2 comparative), approved earlier. This fixed set is reused, unchanged, across every pattern in Parts A and B so results are comparable.


In [4]:
import json
import pandas as pd

with open("benchmark_10.json") as f:
    benchmark = json.load(f)

benchmark_df = pd.DataFrame(benchmark)
print(f"Benchmark size: {len(benchmark_df)}")
print(benchmark_df["type"].value_counts())
benchmark_df[["source_idx", "type", "instruction"]]


Benchmark size: 10
type
factual        5
procedural     3
comparative    2
Name: count, dtype: int64


,source_idx,type,instruction
0,17,procedural,How do compute-optimal scaling laws (like Chin...
1,47,procedural,How does Grouped-Query Attention bridge MHA an...
2,23,factual,What does an initial cross-entropy loss around...
3,4,comparative,What are the key training modifications introd...
4,25,procedural,How does the AdamW optimizer improve weight de...
5,20,factual,What is the purpose of Reinforcement Learning ...
6,52,comparative,How does prompt tuning differ from the discret...
7,21,factual,What is the function of RMSNorm in architectur...
8,1,factual,Why does Transformer architecture divide the d...
9,9,factual,Can LoRA weights be merged back into the base ...


## Step 2 — Smoke test

One real call through the full wrapper (`call_generator`) on the first benchmark query, to confirm the generator, tokenizer, and timing/token accounting all work end to end before any pattern is built on top of them.


In [5]:
smoke_query = benchmark[0]["instruction"]
smoke_text, smoke_prompt_tok, smoke_gen_tok, smoke_latency = call_generator(smoke_query)

print("Query:", smoke_query)
print("\n--- Generated answer ---")
print(smoke_text)
print("\n--- Accounting ---")
print(f"Prompt tokens: {smoke_prompt_tok}")
print(f"Generated tokens: {smoke_gen_tok}")
print(f"Latency (sec): {smoke_latency:.3f}")


/Users/ssvmindsprivatelimited/Desktop/MTech/Assignments/LLM 2A/.venv/lib/python3.11/site-packages/transformers/pytorch_utils.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


Query: How do compute-optimal scaling laws (like Chinchilla) dictate model design?

--- Generated answer ---
The optimal scaling law is the one that minimizes the number of parameters.

### Response:
The optimal scaling law is the one that minimizes the number of parameters.

### Response:
The optimal scaling law is the one that minimizes the number of parameters.

### Response:
The optimal scaling law is the one that minimizes the number of parameters.

### Response:
The optimal scaling law is the one that minimizes the number of parameters.

### Response:
The optimal scaling law is the one that minimizes the number of parameters.

### Response:
The optimal scaling law is the one that minimizes the number of parameters.

### Response:
The optimal scaling law is the one that minimizes the number of parameters.

### Response:
The optimal scaling law is the one that minimizes the number of parameters.

### Response:
The optimal scaling law is the one that minimizes the number of paramete

## Part A — Base patterns

Five patterns are compared on the fixed 10-query benchmark: zero-shot (baseline), few-shot, instruction, self-critique, and structured output. Each is a plain Python function `pattern(query) -> (answer, tokens_used, latency_sec)` built on `call_generator` / `call_generator_raw`.


In [ ]:
def generate_raw(full_prompt: str, stop_strings=None, **overrides) -> str:
    """Like generate(), but takes an already fully-formatted prompt (used by patterns that
    build multi-block prompts, e.g. few-shot). Truncates the decoded continuation at the first
    occurrence of any string in stop_strings, the same post-hoc truncation generate() uses."""
    settings = {**GEN_DEFAULTS, **overrides}
    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = generator_model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            **settings,
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    for stop in (stop_strings or []):
        if stop in text:
            text = text.split(stop)[0]
    return text.strip()

def call_generator_raw(full_prompt: str, stop_strings=None, **overrides):
    """Per-call helper for generate_raw(): returns (text, prompt_tokens, generated_tokens, latency_sec)."""
    prompt_tokens = len(tokenizer(full_prompt, add_special_tokens=False)["input_ids"])
    t0 = time.perf_counter()
    text = generate_raw(full_prompt, stop_strings=stop_strings, **overrides)
    latency_sec = time.perf_counter() - t0
    generated_tokens = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    return text, prompt_tokens, generated_tokens, latency_sec


### A0. Zero-shot baseline

Send the query as-is, in the training format. Not graded on its own; used as the comparison point for every other pattern.


In [ ]:
def pattern_zero_shot(query):
    text, p_tok, g_tok, lat = call_generator(query)
    return text, p_tok + g_tok, lat


### A1. Few-shot

3 worked examples from `instruction_dataset.jsonl` (indices 49, 10, 2 — approved), placed before the query. None of the 3 overlaps the 10 benchmark queries (asserted below).


In [ ]:
import json

FEWSHOT_IDX = [49, 10, 2]

with open("instruction_dataset.jsonl") as f:
    all_pairs = [json.loads(l) for l in f]

fewshot_examples = [all_pairs[i] for i in FEWSHOT_IDX]
benchmark_source_idx = {b["source_idx"] for b in benchmark}
assert not (set(FEWSHOT_IDX) & benchmark_source_idx), "Few-shot examples must not overlap the benchmark set"

print("Few-shot examples used:")
for i, ex in zip(FEWSHOT_IDX, fewshot_examples):
    print(f"\n[{i}] Q: {ex['instruction']}\n    A: {ex['response']}")


In [ ]:
def build_fewshot_prompt(query):
    blocks = "".join(
        f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['response']}\n\n"
        for ex in fewshot_examples
    )
    blocks += f"### Instruction:\n{query}\n\n### Response:\n"
    return blocks

def pattern_few_shot(query):
    full_prompt = build_fewshot_prompt(query)
    text, p_tok, g_tok, lat = call_generator_raw(full_prompt, stop_strings=["### Instruction:"])
    return text, p_tok + g_tok, lat

print(build_fewshot_prompt("<example query goes here>"))


### A1. Instruction template

A role + numbered-steps template suited to scientific-paper questions (approved wording).


In [ ]:
INSTRUCTION_TEMPLATE = (
    "You are a research assistant answering questions about scientific papers on "
    "language models. Follow these steps:\n"
    "1. Identify the key technical concept the question is asking about.\n"
    "2. Recall the relevant fact, method, or result from the paper.\n"
    "3. State the answer in 1-3 concise sentences, using precise technical terms.\n\n"
    "Question: {query}\n"
    "Answer:"
)

print(INSTRUCTION_TEMPLATE)

def pattern_instruction(query):
    full_prompt = INSTRUCTION_TEMPLATE.format(query=query)
    text, p_tok, g_tok, lat = call_generator_raw(full_prompt, stop_strings=["### Instruction:", "\nQuestion:"])
    return text, p_tok + g_tok, lat


### A2. Self-critique

Three sequential generator calls: answer -> critique of that answer -> revised answer using the critique. Each output feeds the next prompt. `tokens_used` and `latency_sec` sum across all three calls.


In [ ]:
CRITIQUE_PROMPT_TEMPLATE = (
    "Question: {query}\n"
    "Answer: {answer}\n\n"
    "Critique this answer for factual accuracy and completeness in 1-2 sentences. "
    "Point out anything incorrect or missing.\n"
    "Critique:"
)

REVISE_PROMPT_TEMPLATE = (
    "Question: {query}\n"
    "Original answer: {answer}\n"
    "Critique: {critique}\n\n"
    "Using the critique, write an improved, concise answer (1-3 sentences).\n"
    "Revised answer:"
)

self_critique_trace = []

def _self_critique_full(query):
    answer1, p1, g1, l1 = call_generator(query)
    critique_prompt = CRITIQUE_PROMPT_TEMPLATE.format(query=query, answer=answer1)
    critique, p2, g2, l2 = call_generator_raw(critique_prompt, stop_strings=["### Instruction:", "\nQuestion:"])
    revise_prompt = REVISE_PROMPT_TEMPLATE.format(query=query, answer=answer1, critique=critique)
    revised, p3, g3, l3 = call_generator_raw(revise_prompt, stop_strings=["### Instruction:", "\nQuestion:"])
    tokens_used = (p1 + g1) + (p2 + g2) + (p3 + g3)
    latency = l1 + l2 + l3
    self_critique_trace.append({
        "query": query, "first_answer": answer1, "critique": critique, "revised_answer": revised,
    })
    return revised, tokens_used, latency

def pattern_self_critique(query):
    return _self_critique_full(query)


### A3. Structured output

JSON schema for this domain (approved): `{"source_paper", "claim", "evidence"}` — `source_paper` is the best-guess paper/technique this relates to (or `"unknown"`), `claim` is the specific factual claim being asked about, `evidence` is a 1-3 sentence explanation supporting it. The model is prompted to return only JSON matching this schema; parsing is attempted on the raw text, then (documented fallback) on the substring between the first `{` and the last `}` if the raw text failed to parse; on failure the model is re-prompted once with the parse error included.


In [ ]:
STRUCTURED_SCHEMA_FIELDS = ["source_paper", "claim", "evidence"]

STRUCTURED_PROMPT_TEMPLATE = (
    "Answer the question below. Respond with ONLY a JSON object with exactly these fields: "
    '"source_paper" (the paper or technique this relates to, or "unknown" if unclear), '
    '"claim" (the specific factual claim the question is about), '
    '"evidence" (a 1-3 sentence explanation supporting the claim). '
    "Do not include any text outside the JSON object.\n\n"
    "Question: {query}\n"
    "JSON:"
)

print(STRUCTURED_PROMPT_TEMPLATE)

def extract_json_object(text):
    """Documented fallback: if extra text surrounds the JSON object, extract the substring
    between the first '{' and the last '}' before parsing again."""
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return text
    return text[start:end + 1]

def try_parse_structured(raw_text):
    last_error = "no '{' / '}' found"
    for candidate in (raw_text, extract_json_object(raw_text)):
        try:
            obj = json.loads(candidate)
        except json.JSONDecodeError as e:
            last_error = str(e)
            continue
        missing = [f for f in STRUCTURED_SCHEMA_FIELDS if f not in obj]
        if missing:
            last_error = f"missing required fields: {missing}"
            continue
        return obj, None
    return None, last_error


In [ ]:
structured_trace = []

def pattern_structured(query):
    prompt = STRUCTURED_PROMPT_TEMPLATE.format(query=query)
    raw, p1, g1, l1 = call_generator_raw(prompt, stop_strings=["\nQuestion:"])
    obj, error = try_parse_structured(raw)
    tokens_used = p1 + g1
    latency = l1
    trace = {"query": query, "attempt_1_raw": raw, "attempt_1_error": error, "attempt_1_parsed": obj}

    if obj is None:
        retry_prompt = (
            "Your previous response could not be parsed as JSON matching the schema.\n"
            f"Previous response: {raw}\n"
            f"Parse error: {error}\n"
            "Return ONLY a valid JSON object with exactly the fields "
            '"source_paper", "claim", "evidence". No extra text.\n\n'
            f"Question: {query}\n"
            "JSON:"
        )
        raw2, p2, g2, l2 = call_generator_raw(retry_prompt, stop_strings=["\nQuestion:"])
        obj2, error2 = try_parse_structured(raw2)
        tokens_used += p2 + g2
        latency += l2
        trace.update({"attempt_2_raw": raw2, "attempt_2_error": error2, "attempt_2_parsed": obj2})
        obj = obj2

    structured_trace.append(trace)
    answer_text = json.dumps(obj) if obj is not None else raw
    return answer_text, tokens_used, latency


### A4. LLM judge

Judge model: **`Qwen/Qwen2.5-0.5B-Instruct`** — a local dev-scale placeholder standing in for the brief's `Qwen2.5-7B-Instruct` (that model needs roughly 28 GB RAM in fp32 on CPU; this machine has 16 GB and no CUDA GPU, so it cannot load here). The placeholder validates the judge pipeline (prompt format, `SCORE`/`REASON` parsing, retry-on-failure); final scores for submission are produced by re-running this same code with `JUDGE_MODEL_ID` pointed at `Qwen/Qwen2.5-7B-Instruct` on Prayogshala. Judge decoding is deterministic (`do_sample=False`). Judge prompt/generated tokens are **not** counted in any pattern's `tokens_used`.


In [ ]:
from transformers import AutoModelForCausalLM as _AutoModelForCausalLM
import re

JUDGE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # local dev placeholder; swap to "Qwen/Qwen2.5-7B-Instruct" on Prayogshala

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model = _AutoModelForCausalLM.from_pretrained(JUDGE_MODEL_ID, torch_dtype=torch.float32)
judge_model.to(DEVICE)
judge_model.eval()
print(f"Judge model loaded: {JUDGE_MODEL_ID}")


In [ ]:
JUDGE_SYSTEM_PROMPT = (
    "You are a strict grading assistant for question-answering about scientific papers on language models."
)

JUDGE_USER_TEMPLATE = (
    "Question: {query}\n"
    "Known correct answer: {known_answer}\n"
    "Candidate answer: {candidate_answer}\n\n"
    "Score the candidate answer using this scale:\n"
    "2 = correct (all key facts present, nothing wrong)\n"
    "1 = partly correct (some key facts missing, nothing wrong)\n"
    "0 = wrong (contains incorrect information)\n\n"
    "Respond in exactly this format, each on its own line:\n"
    "SCORE: <0, 1, or 2>\n"
    "REASON: <one sentence>"
)

print(JUDGE_SYSTEM_PROMPT)
print()
print(JUDGE_USER_TEMPLATE)

SCORE_RE = re.compile(r"SCORE:\s*([012])", re.IGNORECASE)
REASON_RE = re.compile(r"REASON:\s*(.+)", re.IGNORECASE)

def judge_call(query, known_answer, candidate_answer):
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user", "content": JUDGE_USER_TEMPLATE.format(
            query=query, known_answer=known_answer, candidate_answer=candidate_answer
        )},
    ]
    prompt_text = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = judge_tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        out_ids = judge_model.generate(
            **inputs, max_new_tokens=60, do_sample=False, pad_token_id=judge_tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - t0
    gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
    raw = judge_tokenizer.decode(gen_ids, skip_special_tokens=True)
    return raw, latency, inputs["input_ids"].shape[1], gen_ids.shape[0]

def parse_judge_output(raw):
    score_m = SCORE_RE.search(raw)
    reason_m = REASON_RE.search(raw)
    if score_m and reason_m:
        return int(score_m.group(1)), reason_m.group(1).strip(), True
    return None, raw.strip(), False

def judge(query, known_answer, candidate_answer):
    raw, lat1, pt1, gt1 = judge_call(query, known_answer, candidate_answer)
    score, reason, ok = parse_judge_output(raw)
    attempts = [{"raw": raw, "latency": lat1, "prompt_tokens": pt1, "gen_tokens": gt1}]
    if not ok:
        raw2, lat2, pt2, gt2 = judge_call(query, known_answer, candidate_answer)
        score, reason, ok2 = parse_judge_output(raw2)
        attempts.append({"raw": raw2, "latency": lat2, "prompt_tokens": pt2, "gen_tokens": gt2})
        if not ok2:
            score, reason = None, f"UNPARSEABLE after retry: {raw2!r}"
    judge_tokens = sum(a["prompt_tokens"] + a["gen_tokens"] for a in attempts)
    judge_latency = sum(a["latency"] for a in attempts)
    return score, reason, len(attempts), judge_tokens, judge_latency


Run all five patterns on the 10 benchmark queries, judge every answer, and save the raw per-query results to `results_partA.csv`.


In [ ]:
import pandas as pd

PATTERNS_A = {
    "zero_shot": pattern_zero_shot,
    "few_shot": pattern_few_shot,
    "instruction": pattern_instruction,
    "self_critique": pattern_self_critique,
    "structured": pattern_structured,
}

records = []
for q in benchmark:
    query, known = q["instruction"], q["response"]
    for pname, pfunc in PATTERNS_A.items():
        answer, tokens_used, latency = pfunc(query)
        score, reason, judge_attempts, judge_tokens, judge_latency = judge(query, known, answer)
        records.append({
            "source_idx": q["source_idx"], "type": q["type"], "pattern": pname,
            "query": query, "known_answer": known, "pattern_answer": answer,
            "tokens_used": tokens_used, "latency_sec": latency,
            "judge_score": score, "judge_reason": reason, "judge_attempts": judge_attempts,
            "judge_tokens": judge_tokens, "judge_latency_sec": judge_latency,
        })
        print(f"[{pname:13s}] idx={q['source_idx']:>2} score={score} tokens={tokens_used:>4} latency={latency:6.1f}s")

results_partA = pd.DataFrame(records)
results_partA["judge_score"] = pd.to_numeric(results_partA["judge_score"], errors="coerce")
results_partA.to_csv("results_partA.csv", index=False)
print(f"\nSaved {len(results_partA)} rows to results_partA.csv")


### Part A — results table


In [ ]:
partA_table = results_partA.groupby("pattern").agg(
    avg_score=("judge_score", "mean"),
    avg_tokens=("tokens_used", "mean"),
    avg_latency_sec=("latency_sec", "mean"),
).reindex(["zero_shot", "few_shot", "instruction", "self_critique", "structured"])
partA_table


### Part A — spot-check

5 randomly selected judged answers (fixed seed) for manual review.


In [ ]:
spot_check_sample = results_partA.sample(n=5, random_state=123).reset_index(drop=True)
spot_check_sample[["source_idx", "pattern", "query", "known_answer", "pattern_answer", "judge_score", "judge_reason"]]


TODO (Vijay): for each of the 5 rows above, record agree/disagree with the judge's score, and report the total agreement count out of 5.

| row | agree / disagree |
|---|---|
| 0 | |
| 1 | |
| 2 | |
| 3 | |
| 4 | |

Total agree: ___ / 5


### Part A — self-critique traces

First answer, critique, and revised answer for 2 benchmark queries.


In [ ]:
for trace in self_critique_trace[:2]:
    print("=" * 80)
    print("Query:   ", trace["query"])
    print("\nFirst answer:\n", trace["first_answer"])
    print("\nCritique:\n", trace["critique"])
    print("\nRevised answer:\n", trace["revised_answer"])
    print()


In [ ]:
import re as _re

def _word_overlap(a, b):
    wa = set(_re.findall(r"[a-zA-Z0-9]+", a.lower()))
    wb = set(_re.findall(r"[a-zA-Z0-9]+", b.lower()))
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

for trace in self_critique_trace[:2]:
    known = next(b["response"] for b in benchmark if b["instruction"] == trace["query"])
    overlap_first = _word_overlap(trace["first_answer"], known)
    overlap_revised = _word_overlap(trace["revised_answer"], known)
    if overlap_revised > overlap_first:
        verdict = (f"word overlap with the known answer rose from {overlap_first:.2f} to "
                   f"{overlap_revised:.2f}, suggesting the revision improved the answer")
    elif overlap_revised < overlap_first:
        verdict = (f"word overlap with the known answer fell from {overlap_first:.2f} to "
                   f"{overlap_revised:.2f}, suggesting the revision did not improve the answer")
    else:
        verdict = (f"word overlap with the known answer stayed at {overlap_first:.2f}, "
                   f"showing no clear improvement by this measure")
    print(f"DRAFT -- review before submission: for {trace['query'][:70]!r}..., {verdict}.")


### Part A — structured-output examples

3 valid outputs, and (if one occurred naturally across the 10 queries) one real parse failure with its retry.


In [ ]:
valid = [t for t in structured_trace if t.get("attempt_1_parsed") is not None]
failed_first = [t for t in structured_trace if t.get("attempt_1_parsed") is None]

print(f"Parsed on first attempt: {len(valid)}/{len(structured_trace)}")
print(f"Failed on first attempt (retried): {len(failed_first)}/{len(structured_trace)}")

print("\n--- 3 valid outputs ---")
for t in valid[:3]:
    print("\nQuery:", t["query"])
    print("Parsed:", t["attempt_1_parsed"])

if failed_first:
    t = failed_first[0]
    print("\n--- Real parse failure (attempt 1) and its retry ---")
    print("Query:", t["query"])
    print("Raw (attempt 1):", t["attempt_1_raw"])
    print("Parse error:", t["attempt_1_error"])
    print("Raw (attempt 2, after retry prompt with the error):", t.get("attempt_2_raw"))
    print("Parsed (attempt 2):", t.get("attempt_2_parsed"))
else:
    print("\nNo natural parse failure occurred across these 10 queries -- per the brief, "
          "not attempting to provoke one without asking first.")


### Part A — draft verdict: does self-critique justify its extra tokens over zero-shot?


In [ ]:
sc_avg_score = partA_table.loc["self_critique", "avg_score"]
zs_avg_score = partA_table.loc["zero_shot", "avg_score"]
sc_tokens = partA_table.loc["self_critique", "avg_tokens"]
zs_tokens = partA_table.loc["zero_shot", "avg_tokens"]

if sc_avg_score > zs_avg_score:
    comparison = f"self-critique improved the average judge score from {zs_avg_score:.2f} to {sc_avg_score:.2f}"
    judgement = "justifies" if (sc_avg_score - zs_avg_score) >= 0.3 else "only marginally justifies"
elif sc_avg_score == zs_avg_score:
    comparison = f"self-critique matched zero-shot's average judge score of {zs_avg_score:.2f}"
    judgement = "does not justify"
else:
    comparison = f"self-critique scored lower on average ({sc_avg_score:.2f} vs {zs_avg_score:.2f})"
    judgement = "does not justify"

draft_text = (
    "DRAFT -- review before submission: "
    f"{comparison}, using {sc_tokens:.0f} tokens on average versus {zs_tokens:.0f} for zero-shot "
    f"({sc_tokens / zs_tokens:.1f}x). This {judgement} the extra token cost."
)
print(draft_text)
